<a href="https://colab.research.google.com/github/xlbrosxl/dashboard/blob/main/dashboard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
import os
import pandas as pd
import numpy as np
import json
import base64
from IPython.display import display, HTML

# ============================================================
# BSC CDAV — DASHBOARD EJECUTIVO INTERACTIVO
# ============================================================

try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

FILE_NAME = 'BSC_CDAV_Dashboard_PowerBI_v5.xlsx'

PERSP_COLORS = {'FINANCIERA':'#3B82F6','CLIENTE':'#10B981','PROCESOS':'#F59E0B','CRECIMIENTO':'#8B5CF6'}
PERSP_ICONS  = {'FINANCIERA':'💰','CLIENTE':'👥','PROCESOS':'⚙️','CRECIMIENTO':'📈'}
SEM_COLORS   = {'VERDE':'#10B981','AMARILLO':'#F59E0B','ROJO':'#EF4444'}

CSS = """
<style>
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@300;400;500;600;700&display=swap');
*{box-sizing:border-box;margin:0;padding:0}
.bsc{font-family:'Inter',sans-serif;background:#0a0a14;color:#f0f0f0;padding:24px;border-radius:16px}
.bsc-header{background:linear-gradient(135deg,#4B0082,#7B2FBE,#9B59F0);padding:24px 30px;border-radius:14px;margin-bottom:20px;display:flex;justify-content:space-between;align-items:center;box-shadow:0 8px 32px rgba(123,47,190,0.3)}
.bsc-header h1{font-size:22px;font-weight:700;color:#fff}
.bsc-header .sub{font-size:12px;opacity:0.85;margin-top:4px;color:#e0d4f5}
.bsc-header .badge{background:rgba(255,255,255,0.2);backdrop-filter:blur(10px);padding:8px 16px;border-radius:20px;font-size:12px;font-weight:600;color:#fff}
.cards-grid{display:grid;grid-template-columns:repeat(5,1fr);gap:14px;margin-bottom:20px}
.s-card{background:#12121f;border-radius:12px;padding:18px;border:1px solid #1e1e35;transition:all 0.3s ease;cursor:default;position:relative;overflow:hidden}
.s-card:hover{border-color:#7B2FBE;transform:translateY(-2px);box-shadow:0 8px 24px rgba(0,0,0,0.4)}
.s-card::before{content:'';position:absolute;top:0;left:0;right:0;height:3px;border-radius:12px 12px 0 0}
.s-card .label{font-size:10px;color:#8888a8;text-transform:uppercase;letter-spacing:1.5px;margin-bottom:8px;font-weight:600}
.s-card .value{font-size:32px;font-weight:700;line-height:1}
.s-card .detail{font-size:11px;color:#6b6b8a;margin-top:6px}
.sec-title{font-size:13px;font-weight:700;color:#9B59F0;text-transform:uppercase;letter-spacing:2px;margin:24px 0 14px;padding-bottom:8px;border-bottom:1px solid #1e1e35;display:flex;align-items:center;gap:8px}
.gauges-grid{display:grid;grid-template-columns:repeat(4,1fr);gap:14px;margin-bottom:20px}
.g-card{background:#12121f;border-radius:12px;padding:20px 16px;text-align:center;border:1px solid #1e1e35;cursor:pointer;transition:all 0.3s ease}
.g-card:hover{border-color:#7B2FBE;transform:translateY(-3px);box-shadow:0 12px 32px rgba(0,0,0,0.5)}
.g-card .g-label{font-size:11px;color:#8888a8;text-transform:uppercase;letter-spacing:1.5px;font-weight:600;margin-bottom:12px}
.g-card .g-val{font-size:28px;font-weight:700;margin:8px 0}
.g-card .g-info{font-size:10px;color:#6b6b8a;margin-top:8px;line-height:1.4}
.g-bar{height:6px;background:#1e1e35;border-radius:3px;margin-top:10px;overflow:hidden}
.g-fill{height:100%;border-radius:3px;transition:width 1.5s ease}
.charts-grid{display:grid;grid-template-columns:1fr 1fr;gap:14px;margin-bottom:20px}
.chart-box{background:#12121f;border-radius:12px;padding:20px;border:1px solid #1e1e35}
.chart-box h3{font-size:13px;font-weight:600;color:#c0c0d8;margin-bottom:14px;display:flex;align-items:center;gap:6px}
.chart-box canvas{width:100%!important;max-height:260px}
.chart-full{grid-column:1/-1}
.kpi-tabs{display:flex;gap:6px;flex-wrap:wrap;margin-bottom:12px}
.kpi-tab{padding:5px 12px;border-radius:16px;font-size:10px;font-weight:600;border:1px solid #2a2a4a;background:#0a0a14;color:#8888a8;cursor:pointer;transition:all 0.2s}
.kpi-tab.active,.kpi-tab:hover{background:#7B2FBE;color:#fff;border-color:#7B2FBE}
.tbl{width:100%;border-collapse:separate;border-spacing:0;border-radius:12px;overflow:hidden}
.tbl th{background:#1a1035;color:#c0b8e8;padding:12px 10px;font-size:10px;text-transform:uppercase;letter-spacing:1px;text-align:left;font-weight:700}
.tbl td{padding:12px 10px;border-bottom:1px solid #1a1a2e;font-size:12px;background:#0e0e1a;color:#d0d0e0;transition:background 0.2s}
.tbl tr.kpi-row{cursor:pointer}
.tbl tr.kpi-row:hover td{background:#18182e}
.tbl .persp-hdr td{background:#141428!important;font-weight:700;font-size:13px;border-bottom:2px solid #2a2a4a;color:#f0f0f0}
.pill{padding:4px 10px;border-radius:12px;font-size:10px;font-weight:700;display:inline-block}
.pill-verde{background:rgba(16,185,129,0.15);color:#10B981}
.pill-amarillo{background:rgba(245,158,11,0.15);color:#F59E0B}
.pill-rojo{background:rgba(239,68,68,0.15);color:#EF4444}
.pbar{width:100%;height:6px;background:#1e1e35;border-radius:3px;overflow:hidden}
.pfill{height:100%;border-radius:3px}
.insights{background:#12121f;border-radius:12px;padding:20px;border:1px solid #1e1e35;border-left:4px solid #7B2FBE;margin-top:20px}
.insights h3{font-weight:700;font-size:14px;color:#9B59F0;margin-bottom:12px}
.alert-item{font-size:12px;margin:6px 0;padding:8px 12px;border-radius:8px;display:flex;align-items:center;gap:8px}
.alert-g{background:rgba(16,185,129,0.08);color:#34D399}
.alert-y{background:rgba(245,158,11,0.08);color:#FBBF24}
.alert-r{background:rgba(239,68,68,0.08);color:#F87171}
.conclusion{margin-top:14px;padding:12px 16px;border-radius:10px;font-weight:700;font-size:13px}
.modal-ov{position:fixed;top:0;left:0;right:0;bottom:0;background:rgba(0,0,0,0.75);backdrop-filter:blur(6px);display:none;align-items:center;justify-content:center;z-index:9999}
.modal-ov.show{display:flex}
.modal-box{background:#12121f;border-radius:16px;padding:28px;max-width:600px;width:90%;max-height:80vh;overflow-y:auto;border:1px solid #2a2a4a;box-shadow:0 24px 64px rgba(0,0,0,0.6);animation:modalIn 0.3s ease}
@keyframes modalIn{from{opacity:0;transform:scale(0.9) translateY(20px)}to{opacity:1;transform:scale(1) translateY(0)}}
.modal-close{position:absolute;top:16px;right:20px;background:none;border:none;color:#888;font-size:24px;cursor:pointer;transition:color 0.2s}
.modal-close:hover{color:#fff}
.modal-box h2{font-size:18px;font-weight:700;margin-bottom:4px;color:#f0f0f0}
.modal-box .m-persp{font-size:11px;font-weight:600;text-transform:uppercase;letter-spacing:1px;margin-bottom:16px}
.modal-box .m-section{margin:16px 0;padding:14px;background:#0a0a14;border-radius:10px;border:1px solid #1e1e35}
.modal-box .m-section h4{font-size:11px;color:#8888a8;text-transform:uppercase;letter-spacing:1px;margin-bottom:8px;font-weight:700}
.modal-box .m-row{display:flex;justify-content:space-between;align-items:center;padding:6px 0}
.modal-box .m-label{font-size:12px;color:#8888a8}
.modal-box .m-val{font-size:14px;font-weight:700}
.modal-box .m-strategy{font-size:12px;color:#a0a0b8;line-height:1.6;margin-top:8px}
.modal-box canvas{margin-top:12px;max-height:180px}
@keyframes fadeUp{from{opacity:0;transform:translateY(16px)}to{opacity:1;transform:translateY(0)}}
.anim{animation:fadeUp 0.5s ease both}
.anim-d1{animation-delay:0.05s}.anim-d2{animation-delay:0.1s}.anim-d3{animation-delay:0.15s}.anim-d4{animation-delay:0.2s}.anim-d5{animation-delay:0.25s}
</style>
"""

JS = """
<script src="https://cdn.jsdelivr.net/npm/chart.js@4.4.0/dist/chart.umd.min.js"></script>
<script>
(function(){
const D=window.__BSC__;
const kpis=D.kpis, persps=D.persps, stats=D.stats;
const PC=D.perspColors, PI=D.perspIcons, SC=D.semColors;
let modalChart=null;

// ===== CHARTS =====
function waitChart(cb){if(typeof Chart!=='undefined')cb();else setTimeout(()=>waitChart(cb),150)}

waitChart(function(){
  Chart.defaults.color='#a0a0b8';
  Chart.defaults.font.family='Inter,sans-serif';

  // 1. DONUT with click handler
  const donutChart=new Chart(document.getElementById('chart-donut'),{
    type:'doughnut',
    data:{labels:['Cumplen','En Riesgo','Críticos'],datasets:[{
      data:[stats.n_verde,stats.n_amarillo,stats.n_rojo],
      backgroundColor:['#10B981','#F59E0B','#EF4444'],
      borderWidth:0,hoverOffset:8
    }]},
    options:{responsive:true,maintainAspectRatio:true,cutout:'70%',
      onClick:function(evt,elements){
        if(!elements.length)return;
        const idx=elements[0].index;
        const sem=['VERDE','AMARILLO','ROJO'][idx];
        const titles=['✅ KPIs que Cumplen Meta','⚠️ KPIs en Riesgo','❌ KPIs Críticos'][idx];
        const colors=['#10B981','#F59E0B','#EF4444'][idx];
        const filtered=kpis.filter(k=>k.semaforo===sem);
        let h='<h2 style="color:'+colors+'">'+titles+'</h2>';
        h+='<div class="m-persp" style="color:'+colors+'">'+filtered.length+' de '+kpis.length+' indicadores</div>';
        filtered.forEach(function(k){
          const av=Math.round(k.avance*100);
          const gap=((k.meta_2026-k.real_2026)*100).toFixed(1);
          const pcol=PC[k.perspectiva]||'#888';
          h+='<div class="m-section" style="cursor:pointer" onclick="__openModal('+kpis.indexOf(k)+')">';
          h+='<div style="display:flex;justify-content:space-between;align-items:center;margin-bottom:8px">';
          h+='<span style="font-weight:700;color:#f0f0f0;font-size:13px">'+k.kpi+'</span>';
          h+='<span class="pill pill-'+k.semaforo.toLowerCase()+'">'+av+'%</span></div>';
          h+='<div style="display:flex;gap:16px;margin-bottom:8px">';
          h+='<span class="m-label">Meta: '+(k.meta_2026*100).toFixed(1)+'%</span>';
          h+='<span class="m-label">Real: <b style="color:'+colors+'">'+(k.real_2026*100).toFixed(1)+'%</b></span>';
          if(sem!=='VERDE')h+='<span class="m-label">Brecha: <b style="color:#F87171">'+gap+'pp</b></span>';
          h+='</div>';
          h+='<div style="margin-bottom:6px"><div class="pbar" style="height:6px"><div class="pfill" style="width:'+Math.min(av,100)+'%;background:'+colors+'"></div></div></div>';
          h+='<div style="font-size:11px;color:#6b6b8a"><b style="color:'+pcol+'">'+k.perspectiva+'</b> · '+k.responsable+'</div>';
          h+='</div>';
        });
        modalBody.innerHTML=h;
        overlay.classList.add('show');
      },
      plugins:{legend:{position:'bottom',labels:{padding:16,usePointStyle:true,pointStyle:'circle',font:{size:11}}},
        tooltip:{backgroundColor:'#1a1a2e',titleColor:'#f0f0f0',bodyColor:'#d0d0e0',borderColor:'#2a2a4a',borderWidth:1,padding:12,cornerRadius:8}
      }
    }
  });

  // 2. HORIZONTAL BAR
  const sortedKpis=[...kpis].sort((a,b)=>a.avance-b.avance);
  new Chart(document.getElementById('chart-bar'),{
    type:'bar',
    data:{labels:sortedKpis.map(k=>k.kpi.length>35?k.kpi.substring(0,35)+'...':k.kpi),
      datasets:[{data:sortedKpis.map(k=>Math.round(k.avance*100)),backgroundColor:sortedKpis.map(k=>SC[k.semaforo]||'#888'),borderRadius:4,borderSkipped:false,barThickness:18}]},
    options:{indexAxis:'y',responsive:true,maintainAspectRatio:false,
      scales:{x:{grid:{color:'#1e1e35'},ticks:{callback:v=>v+'%',font:{size:10}},max:120},y:{grid:{display:false},ticks:{font:{size:10}}}},
      plugins:{legend:{display:false},tooltip:{backgroundColor:'#1a1a2e',titleColor:'#f0f0f0',bodyColor:'#d0d0e0',borderColor:'#2a2a4a',borderWidth:1,padding:12,cornerRadius:8,
        callbacks:{label:function(ctx){const k=sortedKpis[ctx.dataIndex];return['Avance: '+Math.round(k.avance*100)+'%','Meta: '+(k.meta_2026*100).toFixed(1)+'%','Real: '+(k.real_2026*100).toFixed(1)+'%','Resp: '+k.responsable]}}
      }}
    }
  });

  // 3. RADAR
  new Chart(document.getElementById('chart-radar'),{
    type:'radar',
    data:{labels:persps.map(p=>PI[p.nombre]+' '+p.nombre),datasets:[
      {label:'Meta (100%)',data:[100,100,100,100],borderColor:'rgba(155,89,240,0.5)',backgroundColor:'rgba(155,89,240,0.05)',borderDash:[5,5],borderWidth:2,pointRadius:3,pointBackgroundColor:'#9B59F0'},
      {label:'Avance Real',data:persps.map(p=>p.avance*100),borderColor:'#3B82F6',backgroundColor:'rgba(59,130,246,0.15)',borderWidth:2,pointRadius:5,pointBackgroundColor:persps.map(p=>PC[p.nombre])}
    ]},
    options:{responsive:true,maintainAspectRatio:true,
      scales:{r:{grid:{color:'#1e1e35'},angleLines:{color:'#1e1e35'},ticks:{display:false},suggestedMin:0,suggestedMax:100,pointLabels:{font:{size:10,weight:'600'},color:'#c0c0d8'}}},
      plugins:{legend:{position:'bottom',labels:{padding:16,usePointStyle:true,font:{size:11}}},tooltip:{backgroundColor:'#1a1a2e',titleColor:'#f0f0f0',bodyColor:'#d0d0e0',borderColor:'#2a2a4a',borderWidth:1,padding:12,cornerRadius:8}}
    }
  });

  // 4. LINE
  const lineChart=new Chart(document.getElementById('chart-line'),{
    type:'line',data:{labels:['2025','2026','2027','2028','2029'],datasets:[]},
    options:{responsive:true,maintainAspectRatio:true,
      scales:{x:{grid:{color:'#1e1e35'}},y:{grid:{color:'#1e1e35'},ticks:{callback:v=>v+'%'}}},
      plugins:{legend:{position:'bottom',labels:{padding:16,usePointStyle:true,font:{size:11}}},
        tooltip:{backgroundColor:'#1a1a2e',titleColor:'#f0f0f0',bodyColor:'#d0d0e0',borderColor:'#2a2a4a',borderWidth:1,padding:12,cornerRadius:8,callbacks:{label:function(ctx){return ctx.dataset.label+': '+ctx.parsed.y.toFixed(1)+'%'}}}
      },interaction:{intersect:false,mode:'index'}
    }
  });
  function updateLine(idx){
    const k=kpis[idx],col=PC[k.perspectiva]||'#888';
    lineChart.data.datasets=[
      {label:'Meta PETI',data:[k.base_2025*100,k.meta_2026*100,k.meta_2027*100,k.meta_2028*100,k.meta_2029*100],borderColor:'rgba(155,89,240,0.6)',backgroundColor:'rgba(155,89,240,0.08)',borderDash:[6,4],borderWidth:2,pointRadius:4,pointBackgroundColor:'#9B59F0',fill:true,tension:0.3},
      {label:'Real 2026',data:[k.base_2025*100,k.real_2026*100,null,null,null],borderColor:col,backgroundColor:col+'22',borderWidth:3,pointRadius:6,pointBackgroundColor:col,pointBorderColor:'#fff',pointBorderWidth:2,fill:false,tension:0.3}
    ];
    lineChart.update();
    document.querySelectorAll('.kpi-tab').forEach((t,i)=>{t.classList.toggle('active',i===idx)});
  }
  document.querySelectorAll('.kpi-tab').forEach((tab,i)=>{tab.addEventListener('click',()=>updateLine(i))});
  updateLine(0);
});

// ===== MODAL =====
const overlay=document.getElementById('bsc-modal-ov');
const modalBody=document.getElementById('bsc-modal-body');

window.__openModal=function(idx){
  const k=kpis[idx],col=PC[k.perspectiva]||'#888',scol=SC[k.semaforo]||'#888';
  const av=Math.round(k.avance*100),bw=Math.min(Math.max(av,0),100);

  let h='<h2>'+k.kpi+'</h2>';
  h+='<div class="m-persp" style="color:'+col+'">'+PI[k.perspectiva]+' '+k.perspectiva+' · OBJ-'+k.n_objetivo+'</div>';
  h+='<div class="m-section"><h4>Objetivo Estratégico</h4><div class="m-strategy">'+k.objetivo+'</div></div>';
  h+='<div class="m-section"><h4>Estrategia</h4><div class="m-strategy">'+k.estrategia+'</div></div>';
  h+='<div class="m-section"><h4>Indicadores Clave</h4>';
  h+='<div class="m-row"><span class="m-label">Meta 2026</span><span class="m-val">'+(k.meta_2026*100).toFixed(1)+'%</span></div>';
  h+='<div class="m-row"><span class="m-label">Real 2026</span><span class="m-val" style="color:'+scol+'">'+(k.real_2026*100).toFixed(1)+'%</span></div>';
  h+='<div class="m-row"><span class="m-label">Avance vs Meta</span><span class="m-val" style="color:'+scol+'">'+av+'%</span></div>';
  h+='<div style="margin-top:8px"><div class="pbar" style="height:8px"><div class="pfill" style="width:'+bw+'%;background:'+scol+'"></div></div></div>';
  h+='<div class="m-row"><span class="m-label">Estado</span><span class="pill pill-'+k.semaforo.toLowerCase()+'">'+k.estado+'</span></div></div>';
  h+='<div class="m-row"><span class="m-label">Verde ≥</span><span class="m-val" style="color:#10B981">'+(k.umbral_verde*100).toFixed(1)+'%</span></div>';
  h+='<div class="m-row"><span class="m-label">Amarillo ≥</span><span class="m-val" style="color:#F59E0B">'+(k.umbral_amarillo*100).toFixed(1)+'%</span></div></div>';

  h+='<div class="m-section"><h4>Evolución Multianual</h4><canvas id="modal-chart" height="160"></canvas></div>';
  h+='<div class="m-section"><h4>Responsable</h4><div style="font-size:13px;font-weight:600;color:#d0d0e0">'+k.responsable+'</div></div>';

  modalBody.innerHTML=h;
  overlay.classList.add('show');
  setTimeout(function(){
    if(modalChart){modalChart.destroy();modalChart=null}
    const ctx=document.getElementById('modal-chart');
    if(ctx&&typeof Chart!=='undefined'){
      modalChart=new Chart(ctx,{type:'line',
        data:{labels:['Base 2025','Meta 2026','Meta 2027','Meta 2028','Meta 2029'],datasets:[
          {label:'Meta PETI',data:[k.base_2025*100,k.meta_2026*100,k.meta_2027*100,k.meta_2028*100,k.meta_2029*100],borderColor:'#9B59F0',borderDash:[5,3],borderWidth:2,pointRadius:4,pointBackgroundColor:'#9B59F0',fill:false,tension:0.3},
          {label:'Real',data:[k.base_2025*100,k.real_2026*100,null,null,null],borderColor:col,borderWidth:3,pointRadius:6,pointBackgroundColor:col,pointBorderColor:'#fff',pointBorderWidth:2,fill:false}
        ]},
        options:{responsive:true,maintainAspectRatio:true,scales:{x:{grid:{color:'#1e1e35'}},y:{grid:{color:'#1e1e35'},ticks:{callback:v=>v+'%'}}},plugins:{legend:{labels:{font:{size:10},usePointStyle:true,padding:10}}}}
      });
    }
  },100);
};

document.getElementById('bsc-modal-close').addEventListener('click',function(){overlay.classList.remove('show')});
overlay.addEventListener('click',function(e){if(e.target===overlay)overlay.classList.remove('show')});

// Gauge clicks
document.querySelectorAll('.g-card').forEach(function(card){
  card.addEventListener('click',function(){
    const p=this.dataset.persp,pp=persps.find(x=>x.nombre===p);
    if(!pp)return;
    const pKpis=kpis.filter(k=>k.perspectiva===p),col=PC[p]||'#888';
    let h='<h2>'+PI[p]+' '+p+'</h2>';
    h+='<div class="m-persp" style="color:'+col+'">Peso Estratégico: '+(pp.peso*100).toFixed(0)+'% · '+pp.n_kpis+' KPIs</div>';
    h+='<div class="m-section"><h4>Diagnóstico</h4><div class="m-strategy">'+pp.justificacion+'</div></div>';
    h+='<div class="m-section"><h4>KPIs de esta Perspectiva</h4>';
    pKpis.forEach(function(k){
      const av=Math.round(k.avance*100);
      h+='<div style="padding:10px 0;border-bottom:1px solid #1e1e35;cursor:pointer" onclick="__openModal('+kpis.indexOf(k)+')">';
      h+='<div style="display:flex;justify-content:space-between;align-items:center;margin-bottom:4px">';
      h+='<span style="font-size:12px;font-weight:600;color:#d0d0e0">'+k.kpi+'</span>';
      h+='<span class="pill pill-'+k.semaforo.toLowerCase()+'">'+av+'%</span></div>';
      h+='</div>';
    });
    h+='</div>';
    modalBody.innerHTML=h;
    overlay.classList.add('show');
  });
});

// Table row clicks
document.querySelectorAll('.kpi-row').forEach(function(row){
  row.addEventListener('click',function(){__openModal(parseInt(this.dataset.idx))});
});

// Table group toggle
document.querySelectorAll('.persp-hdr').forEach(function(hdr){
  hdr.addEventListener('click',function(){
    const p = this.dataset.toggle;
    const btn = document.getElementById('btn-'+p);
    const rows = document.querySelectorAll('.kpi-row[data-group="'+p+'"]');
    if (!rows.length) return;
    const isHidden = rows[0].style.display === 'none';
    rows.forEach(r => r.style.display = isHidden ? '' : 'none');
    btn.textContent = isHidden ? '-' : '+';
  });
});
})();
</script>
"""

def parse_pct(val):
    """Convierte valor porcentual (string '58%' o float 0.58) a float 0-1."""
    if isinstance(val, str):
        return float(val.strip('%').strip()) / 100
    return float(val)

def run_dashboard():
    # ── 1. CARGAR DATOS ──────────────────────────────────────
    try:
        maestro = pd.read_excel(FILE_NAME, sheet_name='Dataset_Maestro')
        resumen = pd.read_excel(FILE_NAME, sheet_name='Perspectiva_Resumen')
    except Exception as e:
        print(f"❌ Error cargando '{FILE_NAME}': {e}")
        return

    # ── 2. PROCESAR KPIS ─────────────────────────────────────
    kpis = []
    for _, r in maestro.iterrows():
        kpis.append({
            'perspectiva': r['PERSPECTIVA_BSC'],
            'n_objetivo': int(r['N_OBJETIVO']),
            'objetivo': str(r['OBJETIVO_ESTRATEGICO']),
            'estrategia': str(r['ESTRATEGIA']),
            'kpi': str(r['KPI']),
            'unidad': str(r['UNIDAD_KPI']),
            'base_2025': float(r['LINEA_BASE_2025']),
            'meta_2026': float(r['META_2026']),
            'meta_2027': float(r['META_2027']),
            'meta_2028': float(r['META_2028']),
            'meta_2029': float(r['META_2029']),
            'real_2026': float(r['PROYECTADO_REAL_2026']),
            'umbral_verde': float(r['UMBRAL_VERDE_>=']),
            'umbral_amarillo': float(r['UMBRAL_AMARILLO_>=']),
            'avance': float(r['%_AVANCE_vs_META']),
            'semaforo': r['SEMAFORO'],
            'estado': r['ESTADO'],
            'responsable': str(r['RESPONSABLE'])
        })

    # ── 3. PROCESAR PERSPECTIVAS ─────────────────────────────
    persps = []
    for _, r in resumen.iterrows():
        persps.append({
            'nombre': r['PERSPECTIVA_BSC'],
            'n_kpis': int(r['N_KPIs']),
            'avance': parse_pct(r['AVANCE_PROMEDIO']),
            'peso': parse_pct(r['PESO_ESTRATEGICO']),
            'justificacion': str(r['JUSTIFICACION_BALANCE_DIAGNOSTICO'])
        })

    # ── 4. ESTADÍSTICAS ──────────────────────────────────────
    avg = np.mean([k['avance'] for k in kpis]) * 100
    nv = sum(1 for k in kpis if k['semaforo'] == 'VERDE')
    na = sum(1 for k in kpis if k['semaforo'] == 'AMARILLO')
    nr = sum(1 for k in kpis if k['semaforo'] == 'ROJO')
    stats = {'avg_avance': round(avg, 1), 'n_verde': nv, 'n_amarillo': na, 'n_rojo': nr, 'total': len(kpis)}

    # ── 5. CONSTRUIR HTML ────────────────────────────────────
    html = build_dashboard(kpis, persps, stats)

    # Guardar como archivo HTML standalone
    html_file = 'dashboard_bsc.html'
    with open(html_file, 'w', encoding='utf-8') as f:
        f.write(html)
    print(f'✅ Dashboard guardado en: {html_file}')

    # Abrir en pestaña nueva del navegador
    b64 = base64.b64encode(html.encode('utf-8')).decode('utf-8')
    js_open = f'<script>window.open("data:text/html;base64,{b64}","_blank");</script>'
    display(HTML(js_open))
    print('🌐 Se abrió el dashboard en una nueva pestaña.')
    print('💡 Si no se abrió, descarga el archivo dashboard_bsc.html y ábrelo en tu navegador.')

    # También ofrecer descarga en Colab
    if IN_COLAB:
        try:
            from google.colab import files as colab_files
            colab_files.download(html_file)
        except:
            pass

def build_dashboard(kpis, persps, stats):
    kj = json.dumps(kpis, ensure_ascii=False)
    pj = json.dumps(persps, ensure_ascii=False)
    sj = json.dumps(stats)

    # Data injection
    data_script = '<script>window.__BSC__={kpis:' + kj + ',persps:' + pj + ',stats:' + sj
    data_script += ',perspColors:' + json.dumps(PERSP_COLORS)
    data_script += ',perspIcons:' + json.dumps(PERSP_ICONS)
    data_script += ',semColors:' + json.dumps(SEM_COLORS) + '};</script>'

    avg = stats['avg_avance']
    nv, na, nr = stats['n_verde'], stats['n_amarillo'], stats['n_rojo']

    # ── HEADER ───────────────────────────────────────────────
    b = '<div class="bsc">'
    b += '<div class="bsc-header"><div>'
    b += '<h1>📊 DASHBOARD BSC CDAV — PETI 2026-2029</h1>'
    b += '<div class="sub">Balanced Scorecard · Centro de Atención y Valoración · Plan Estratégico de TI</div>'
    b += '</div><div class="badge">🔄 ACTUALIZADO</div></div>'

    # ── SUMMARY CARDS ────────────────────────────────────────
    avg_col = '#10B981' if avg >= 80 else '#F59E0B' if avg >= 50 else '#EF4444'
    b += '<div class="cards-grid">'
    cards = [
        ('Avance Global', f'{avg:.1f}%', avg_col, f'{stats["total"]} indicadores evaluados', '📊'),
        ('Cumplen Meta', str(nv), '#10B981', f'{nv}/{stats["total"]} KPIs en verde', '✅'),
        ('En Riesgo', str(na), '#F59E0B', f'{na}/{stats["total"]} requieren atención', '⚠️'),
        ('Críticos', str(nr), '#EF4444', f'{nr}/{stats["total"]} necesitan acción urgente' if nr > 0 else 'Sin KPIs críticos', '❌'),
        ('Salud Estratégica', 'BUENA' if avg>=80 else 'MODERADA' if avg>=50 else 'CRÍTICA', avg_col, 'Basado en avance promedio', '🏥'),
    ]
    for i, (label, value, color, detail, icon) in enumerate(cards):
        b += f'<div class="s-card anim anim-d{i+1}" style="--accent:{color}">'
        b += f'<div style="position:absolute;top:0;left:0;right:0;height:3px;background:{color};border-radius:12px 12px 0 0"></div>'
        b += f'<div class="label">{icon} {label}</div>'
        b += f'<div class="value" style="color:{color}">{value}</div>'
        b += f'<div class="detail">{detail}</div></div>'
    b += '</div>'

    # ── PERSPECTIVE GAUGES ───────────────────────────────────
    b += '<div class="sec-title">🎯 Avance por Perspectiva Estratégica</div>'
    b += '<div class="gauges-grid">'
    for p in persps:
        col = PERSP_COLORS.get(p['nombre'], '#888')
        icon = PERSP_ICONS.get(p['nombre'], '📊')
        av = p['avance'] * 100
        bw = min(av, 100)
        b += f'<div class="g-card" data-persp="{p["nombre"]}">'
        b += f'<div class="g-label">{icon} {p["nombre"]}</div>'
        b += f'<div class="g-val" style="color:{col}">{av:.1f}%</div>'
        b += f'<div class="g-bar"><div class="g-fill" style="width:{bw}%;background:linear-gradient(90deg,{col},{col}88)"></div></div>'
        b += f'<div class="g-info">Peso: {p["peso"]*100:.0f}% · {p["n_kpis"]} KPIs<br>📌 Click para ver detalle</div>'
        b += '</div>'
    b += '</div>'

    # ── CHARTS ROW 1: Donut + Radar ──────────────────────────
    b += '<div class="sec-title">📈 Análisis Visual</div>'
    b += '<div class="charts-grid">'
    b += '<div class="chart-box"><h3>🍩 Distribución de Semáforos</h3><canvas id="chart-donut"></canvas></div>'
    b += '<div class="chart-box"><h3>🕸️ Radar Estratégico</h3><canvas id="chart-radar"></canvas></div>'

    # ── CHART ROW 2: Bar (full width) ────────────────────────
    b += '<div class="chart-box chart-full" style="min-height:320px"><h3>📊 Avance por Indicador (Ordenado)</h3><canvas id="chart-bar"></canvas></div>'

    # ── CHART ROW 3: Line (full width with tabs) ─────────────
    b += '<div class="chart-box chart-full"><h3>📈 Evolución Multianual PETI 2025-2029</h3>'
    b += '<div class="kpi-tabs">'
    for i, k in enumerate(kpis):
        pcol = PERSP_COLORS.get(k['perspectiva'], '#888')
        active = ' active' if i == 0 else ''
        short_name = k['kpi'][:40] + ('...' if len(k['kpi']) > 40 else '')
        b += f'<div class="kpi-tab{active}" style="--tab-color:{pcol}">{short_name}</div>'
    b += '</div><canvas id="chart-line"></canvas></div>'
    b += '</div>'

    # ── KPI TABLE ────────────────────────────────────────────
    b += '<div class="sec-title">📋 Detalle de Indicadores — Click en cualquier fila para expandir</div>'
    b += '<table class="tbl"><thead><tr>'
    for th in ['Obj', 'KPI', 'Estrategia', 'Meta', 'Real', 'Progreso', 'Avance', 'Estado', 'Responsable']:
        b += f'<th>{th}</th>'
    b += '</tr></thead><tbody>'

    persp_order = ['FINANCIERA', 'CLIENTE', 'PROCESOS', 'CRECIMIENTO']
    for persp in persp_order:
        p_kpis = [k for k in kpis if k['perspectiva'] == persp]
        if not p_kpis:
            continue
        p_avg = np.mean([k['avance'] for k in p_kpis]) * 100
        col = PERSP_COLORS.get(persp, '#888')
        icon = PERSP_ICONS.get(persp, '📊')
        b += f'<tr class="persp-hdr" data-toggle="{persp}" style="cursor:pointer"><td colspan="9" style="border-left:4px solid {col}">'
        b += f'<span class="toggle-btn" id="btn-{persp}" style="display:inline-block;width:22px;height:22px;line-height:22px;text-align:center;background:{col}22;color:{col};border-radius:6px;font-size:14px;font-weight:700;margin-right:8px;transition:all 0.2s">+</span>'
        b += f'{icon} {persp} — <span style="color:{col}">{p_avg:.1f}%</span>'
        b += f'<span style="font-size:10px;color:#6b6b8a;margin-left:12px">({len(p_kpis)} indicadores)</span></td></tr>'
        for k in p_kpis:
            idx = kpis.index(k)
            av = k['avance'] * 100
            sc = SEM_COLORS.get(k['semaforo'], '#888')
            bw = min(max(av, 0), 100)
            sm = k['semaforo'].lower()
            b += f'<tr class="kpi-row" data-idx="{idx}" data-group="{persp}" style="display:none">'
            b += f'<td style="color:#6b6b8a;font-size:10px;font-weight:600">OBJ-{k["n_objetivo"]}</td>'
            b += f'<td style="font-weight:600;color:#e0e0f0">{k["kpi"]}</td>'
            b += f'<td style="font-size:10px;color:#8888a8">{k["estrategia"]}</td>'
            b += f'<td style="text-align:center;color:#a0a0b8">{k["meta_2026"]*100:.1f}%</td>'
            b += f'<td style="text-align:center;font-weight:700;color:{sc}">{k["real_2026"]*100:.1f}%</td>'
            b += f'<td><div class="pbar"><div class="pfill" style="width:{bw}%;background:{sc}"></div></div></td>'
            b += f'<td style="text-align:center;font-weight:700;color:{sc}">{av:.1f}%</td>'
            b += f'<td><span class="pill pill-{sm}">{k["estado"]}</span></td>'
            b += f'<td style="font-size:10px;color:#8888a8">{k["responsable"]}</td></tr>'
    b += '</tbody></table>'

    # ── INSIGHTS PANEL ───────────────────────────────────────
    b += '<div class="insights"><h3>💡 Insights Estratégicos</h3>'

    # KPIs que cumplen
    verdes = [k for k in kpis if k['semaforo'] == 'VERDE']
    for k in verdes:
        b += f'<div class="alert-item alert-g">✅ <b>{k["kpi"]}</b> — cumple meta ({k["avance"]*100:.0f}%) · {k["responsable"]}</div>'

    # KPIs en riesgo
    amarillos = [k for k in kpis if k['semaforo'] == 'AMARILLO']
    for k in amarillos:
        gap = (k['meta_2026'] - k['real_2026']) * 100
        b += f'<div class="alert-item alert-y">⚠️ <b>{k["kpi"]}</b> — brecha de {gap:.1f}pp vs meta · {k["responsable"]}</div>'

    # KPIs críticos
    rojos = [k for k in kpis if k['semaforo'] == 'ROJO']
    for k in rojos:
        b += f'<div class="alert-item alert-r">🔴 <b>{k["kpi"]}</b> — requiere acción urgente ({k["avance"]*100:.0f}%) · {k["responsable"]}</div>'

    # Diagnóstico por perspectiva
    b += '<div style="margin-top:16px;padding-top:14px;border-top:1px solid #1e1e35">'
    b += '<div style="font-size:12px;font-weight:700;color:#9B59F0;margin-bottom:8px">📌 DIAGNÓSTICO POR PERSPECTIVA</div>'
    for p in persps:
        col = PERSP_COLORS.get(p['nombre'], '#888')
        b += f'<div style="font-size:11px;color:#a0a0b8;margin:6px 0;padding-left:12px;border-left:3px solid {col}">'
        b += f'<b style="color:{col}">{p["nombre"]}</b>: {p["justificacion"]}</div>'
    b += '</div>'


    # Conclusión
    if avg >= 80:
        b += '<div class="conclusion" style="background:rgba(16,185,129,0.1);color:#34D399">✔️ ESTRATEGIA SALUDABLE — El PETI avanza conforme a lo planeado. Mantener esfuerzos en KPIs amarillos.</div>'
    elif avg >= 50:
        b += '<div class="conclusion" style="background:rgba(245,158,11,0.1);color:#FBBF24">⚠️ RIESGO MODERADO — Se requieren acciones correctivas en los indicadores con brecha significativa.</div>'
    else:
        b += '<div class="conclusion" style="background:rgba(239,68,68,0.1);color:#F87171">❌ ALERTA CRÍTICA — La ejecución del PETI está significativamente por debajo de las metas.</div>'
    b += '</div>'

    # ── MODAL ────────────────────────────────────────────────
    b += '<div id="bsc-modal-ov" class="modal-ov"><div class="modal-box" style="position:relative">'
    b += '<button id="bsc-modal-close" class="modal-close">&times;</button>'
    b += '<div id="bsc-modal-body"></div></div></div>'

    b += '</div>'  # close .bsc

    # Envolver en documento HTML completo standalone
    full = '<!DOCTYPE html><html lang="es"><head>'
    full += '<meta charset="UTF-8"><meta name="viewport" content="width=device-width,initial-scale=1.0">'
    full += '<title>Dashboard BSC CDAV — PETI 2026-2029</title>'
    full += '</head><body style="margin:0;padding:0;background:#0a0a14">'
    full += CSS + data_script + b + JS
    full += '</body></html>'
    return full

if __name__ == "__main__":
    run_dashboard()


✅ Dashboard guardado en: dashboard_bsc.html


🌐 Se abrió el dashboard en una nueva pestaña.
💡 Si no se abrió, descarga el archivo dashboard_bsc.html y ábrelo en tu navegador.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>